# ROT ベンチマーク — ローカル推論モデルで `<think>` を測る

自己記述性の水準を10段に振ったデータに対して課題を解かせ、**成果あたりのトークン消費**と
**中間推論（`<think>`）のテキストそのもの**を記録します。

API 経由のモデル（gpt-4o-mini / gpt-4.1-mini / gpt-5.4）は `reasoning_tokens` が 0 で返り、
中間推論の長さを分離できませんでした。推論モデルを自前で回すと、思考が生のテキストで取れます。

既定のモデルは [`allenai/Olmo-3-7B-Think`](https://huggingface.co/allenai/Olmo-3-7B-Think) です。
重み・学習コード・学習データが揃って公開されており、OSI の
[Open Source AI Definition](https://opensource.org/ai/open-source-ai-definition) を満たす
数少ない系統です（Qwen 系などは重みのみ公開で、この定義は満たしません）。

## 実行前に

**ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: A100 GPU** を選んでください。

* 7B は A100 40GB でも動きます。32B を選ぶ場合は 80GB が要ります。
* **タブを開いたままにしてください。** 閉じるとランタイムが切れます。

## 所要時間の目安

| 反復 | 目安 |
| --- | --- |
| `REPEATS=1`（既定） | **約2.5時間**（100実行。うち6セルが試行上限まで回る想定） |
| `REPEATS=2` | 約5時間 |
| `REPEATS=5` | 約12.6時間。ブラウザ経路では現実的でありません |

1試行あたりの生成が平均 10,400 トークン、A100 40GB で約 70 tok/s という実測からの外挿です。
**上限10試行まで完走した実測はまだ無い**ので、下振れの可能性があります。

## 記録されるもの

* `attempt_log[].thinking` — **思考のテキストそのもの**
* `thinking_chars` — その文字数
* `output_capped_attempts` — 生成上限に達した試行数。**到達しても集計から除外しません**
* `fingerprint` — 投げたデータ・タスク・プロンプト・サンプリング設定のハッシュと、clone したコミット


## 1. 設定

変えるならここだけです。既定は参照点のランと同じ設定にしてあります。


In [1]:
MODEL   = 'allenai/Olmo-3-7B-Think'   # 32B にするなら A100 80GB が要る
REPEATS = '1'                          # まずは 1。ばらつきを見るなら増やす
TASKS   = 'task_04,task_06'            # 逃げ道のある課題と、無い課題
SUITE   = 'v3_levels'

MAX_ATTEMPTS      = '10'      # 1タスクあたりの試行上限（API 経由の測定と揃えてある）
MAX_OUTPUT_TOKENS = '32768'   # 1リクエストの生成上限。短く切ると切った位置が測定値を決める
MAX_MODEL_LEN     = 65536     # 思考が発散しても文脈長で落ちないよう広めに取る

TEMPERATURE, TOP_P, SEED = '1.0', '1.0', '20260820'
REPO = 'https://github.com/beachcities/RoT.git'


## 2. 取得と起動

リポジトリを clone し、依存を入れて vLLM を立ち上げます。モデルの読み込みまで含めて5〜10分。

`git clone` にしてあるのは、**どのコミットで回したかが結果の指紋に残る**ためです。
手元のファイルを上げる形だと、リポジトリの版と手元の版がずれます。


In [2]:
import subprocess, os, sys, time, urllib.request, shutil

if not os.path.isdir('/content/RoT'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, '/content/RoT'], check=True)
BENCH = '/content/RoT/benchmark'
print('commit:', subprocess.run(['git', '-C', '/content/RoT', 'rev-parse', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())

# Colab のイメージは torch(CUDA 13.0) と torchaudio(12.8) が食い違っており、
# vLLM の import 経路で落ちる。テキスト推論に torchaudio は要らないので外す。
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchaudio'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai', 'python-dotenv'])
try:
    import vllm
    print('vllm', vllm.__version__)
except ImportError:
    print('vllm を入れます（数分）')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

with open(BENCH + '/.env', 'w') as f:
    f.write('API_KEY=dummy' + chr(10) + 'BASE_URL=http://127.0.0.1:8000/v1' + chr(10))

# reasoning-parser は付けない。使えるかはモデルのトークナイザ次第で、
# 付けなくても </think> は本文に残り、ランナー側が切り出す。
log = open('/content/vllm.log', 'w')
server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--port', '8000',
     '--max-model-len', str(MAX_MODEL_LEN), '--gpu-memory-utilization', '0.90'],
    stdout=log, stderr=subprocess.STDOUT)

for i in range(120):
    time.sleep(10)
    if server.poll() is not None:
        print('サーバが落ちました')
        print(open('/content/vllm.log').read()[-3000:])
        break
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3)
        print('READY after', (i + 1) * 10, 's')
        break
    except Exception:
        pass
else:
    print('まだ読み込み中')
    print(open('/content/vllm.log').read()[-2000:])


commit: c8730383d3e03af6d94f2f412e97ef1c80348e86
vllm を入れます（数分）
READY after 270 s


## 3. 疎通の確認

本実行の前に1セルだけ回して、`<think>` が取れているかを見ます。1分程度。


In [3]:
env = dict(os.environ, PYTHONIOENCODING='utf-8',
           MAX_ATTEMPTS=MAX_ATTEMPTS, REPEATS='1', SUITE=SUITE, PROMPT='p1_baseline',
           TEMPERATURE=TEMPERATURE, TOP_P=TOP_P, SEED=SEED,
           MAX_OUTPUT_TOKENS=MAX_OUTPUT_TOKENS, REQUEST_TIMEOUT='1800', MAX_RETRIES='2')

r = subprocess.run([sys.executable, 'run_benchmark.py', '--models', MODEL,
                    '--conditions', 'l6_codes_doc', '--tasks', 'task_06', '--no-save'],
                   cwd=BENCH, capture_output=True, text=True, env=env)
print(r.stdout[-1500:] or r.stderr[-1500:])


  試行  正答  正答率  平均試行 入力     CoT      出力     総token  ROT/1k
-----------------------------------------------------------------------------------------
l6_codes_doc      1     1     100.0%  1.00     2164     n/a      n/a      3495     0.2861

  反復にわたるばらつき（値は1試行あたり）
condition         正答/n   総token中央 Q1-Q3         最小-最大     ROT/1k中央  ROT/1k範囲
-----------------------------------------------------------------------------------------
l6_codes_doc      1/1      3495        n/a           3495-3495     0.2861      0.2861-0.2861

  解けた試行と解けなかった試行を分けたもの（総トークンは二山になる）
condition         内訳      件数  総token中央 Q1-Q3         最小-最大     試行中央
----------------------------------------------------------------------------------
l6_codes_doc      解けた    1     3495        n/a           3495-3495     1.0

------------------------------------------------------------------------------
読み方の留保
  * 分子は正答/誤答の二値。試論が挙げた成果の測り方のいずれでもない。
  * 総トークンの絶対値はトークナイザ依存。モデル間で比べられるのは比のみ。
  * 自己記述的なデータは記述が増える分だけ入力が長くなる。総トークンの
    比を見るときは、

## 4. 本実行

進捗は `running:` の行で追えます。**2.5時間ほどかかります。**


In [4]:
t0 = time.time()
env = dict(env, REPEATS=REPEATS)
proc = subprocess.Popen([sys.executable, '-u', 'run_benchmark.py',
                         '--models', MODEL, '--tasks', TASKS],
                        cwd=BENCH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=env)
for line in proc.stdout:
    print(line, end='', flush=True)
print(chr(10) + '== 所要 ' + str(round((time.time() - t0) / 60)) + '分 / rc=' + str(proc.wait()) + ' ==')


組 v3_levels / プロンプト p1_baseline: 20 セル x 反復 1 回 = 20 回の実行
running: allenai/Olmo-3-7B-Think / l0_opaque / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l0_opaque / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l1_names / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l1_names / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l2_units_ref / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l2_units_ref / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l3_units_doc / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l3_units_doc / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l4_units_record / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l4_units_record / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l5_codes_ref / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l5_codes_ref / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l6_codes_doc / task_04 [1/1]
running: allenai/Olmo-3-7B-Think / l6_codes_doc / task_06 [1/1]
running: allenai/Olmo-3-7B-Think / l7_codes_record / t

In [5]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

## 5. 結果の回収

`/content/` に結果JSONを置きます。左のファイルペインからダウンロードしてください。
思考のテキストを含むので数MBになります。


In [7]:
import os, glob, shutil, time
from google.colab import drive

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/Colab_Results'
os.makedirs(save_dir, exist_ok=True)

src_files = sorted(glob.glob("/content/RoT/benchmark/results/run_*.json"))
is_partial = False
if not src_files:
    src_files = sorted(glob.glob("/content/RoT/benchmark/results/partial/partial_*.jsonl"))
    if not src_files:
        raise FileNotFoundError("結果ファイルもチェックポイントも見つかりませんでした。")
    print("⚠️ 完了時JSONが無いため、最新のチェックポイントを回収します。")
    is_partial = True

src = src_files[-1]
dst = os.path.join(save_dir, os.path.basename(src))
shutil.copy2(src, dst)

time.sleep(0.5)
size = os.path.getsize(dst)
print(f"{'⚠️ [中断]' if is_partial else '✅ [完了]'} 保存: {dst}")
print(f"📦 {size:,} bytes ({size/1024/1024:.2f} MB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ [完了] 保存: /content/drive/MyDrive/Colab_Results/run_20260823T124213Z.json
📦 2,522,913 bytes (2.41 MB)
